In [ ]:
## This is a sample Attention Rollout file for running the rollout on any image.
'''
Ask ChatGPT to 
We used a vit_small_path16_224 model for training. We are using timm Version: 1.0.27. 
Give a dedicated attention_rollout.py script that:
Loads the trained model
Loads an image
Extracts attention matrices correctly from timm
Computes Attention Rollout
Saves:
- Original image
- Rollout heatmap
- Overlay
'''


In [ ]:
attentions = []

def hook_fn(module,input,output):

    attentions.append(
        output.detach()
    )

for block in model.blocks:

    block.attn.attn_drop.register_forward_hook(
        hook_fn
    )


In [ ]:
image = image.unsqueeze(0).to(DEVICE)

_ = model(image)


In [ ]:
result = torch.eye(
    attentions[0].size(-1)
)

for attention in attentions:

    attention_heads_fused = attention.mean(1)

    attention_heads_fused += torch.eye(
        attention_heads_fused.size(-1)
    ).to(DEVICE)

    attention_heads_fused /= (
        attention_heads_fused.sum(
            dim=-1,
            keepdim=True
        )
    )

    result = torch.matmul(
        attention_heads_fused,
        result
    )
    

In [ ]:
mask = result[0,0,1:]


In [ ]:
mask = mask.reshape(
    14,
    14
)

mask = mask.cpu().numpy()

mask = (
    mask - mask.min()
)/(
    mask.max()-mask.min()
)


In [ ]:
import cv2

mask = cv2.resize(
    mask,
    (224,224)
)


In [ ]:
plt.imshow(original_image)

plt.imshow(
    mask,
    cmap="jet",
    alpha=0.5
)

plt.axis("off")

plt.savefig(
    "attention_rollout.png"
)

plt.show()
